# **TRIỂN KHAI MÔ HÌNH PHÂN TÍCH CẢM XÚC**

Notebook này dùng để test mô hình đã được train và lưu từ file Train_Model.ipynb.


In [ ]:
# Cài đặt thư viện cần thiết (nếu chưa có)
%pip install pyvi underthesea nltk tensorflow keras

import string
from underthesea import word_tokenize, pos_tag
from pyvi import ViTokenizer
import nltk
import re
import urllib.request


In [ ]:
# Định nghĩa các hàm tiền xử lý (giống như trong file train)
def xoa_dau_va_ki_tu_dac_biet(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # removing URL links
    text = re.sub(r"\b\d+\b", "", text) # removing number
    text = re.sub('<.*?>+', '', text) # removing special characters,
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # punctuations
    text = re.sub('\n', '', text)
    text = re.sub('[\u2018\u2019\u201c\u201d\u2026]', '', text)
    return text

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

def to_lower(text):
    text = text.lower()
    return text

def remove_nouns_pronouns(text):
    tagged_words = pos_tag(text)
    filtered_words = [word for word, pos in tagged_words if pos not in ['N', 'P']]
    return ' '.join(filtered_words)

# Tải stopwords từ Vietnamese Stopwords dataset
stopwords_url = 'https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/vietnamese-stopwords.txt'
with urllib.request.urlopen(stopwords_url) as response:
    stopwords = set(line.decode('utf-8').strip() for line in response if line.strip())

def clean_text(text):
    text = xoa_dau_va_ki_tu_dac_biet(text)
    text = to_lower(text)
    remove_punctuation(text)
    words = text.split()
    words = [word for word in words if word not in stopwords]
    text = ' '.join(words)
    text = remove_nouns_pronouns(text)
    text = ViTokenizer.tokenize(text)
    return text


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pickle

# Tải mô hình đã train
model = tf.keras.models.load_model('phan_tich_cam_xuc.keras')

# Tải tokenizer đã lưu
tokenizer_path = 'tokenizer.pkl'
with open(tokenizer_path, 'rb') as f:
    tokenizer = pickle.load(f)


In [ ]:
def predict_sentiment(text):
    """
    Hàm dự đoán cảm xúc từ văn bản đầu vào.
    Trả về: "Phản hồi tích cực" hoặc "Phản hồi tiêu cực" cùng với độ tin cậy.
    """
    text = clean_text(text)
    vocab_size = 20000
    max_length = 300
    sequence = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')
    prediction = model.predict(padded, verbose=0)  # Tắt verbose để không in log
    confidence = prediction[0][0]
    if confidence > 0.5:
        return f"Phản hồi tích cực (độ tin cậy: {confidence:.4f})"
    else:
        return f"Phản hồi tiêu cực (độ tin cậy: {confidence:.4f})"


In [ ]:
# Ví dụ kiểm tra
print("Test 1:", predict_sentiment("Sản phẩm rất tốt, tôi rất hài lòng!"))
print("Test 2:", predict_sentiment("Chất lượng quá tệ, không đáng mua."))
print("Test 3:", predict_sentiment("Dịch vụ khách hàng tuyệt vời, sẽ mua lại."))
print("Test 4:", predict_sentiment("Giao hàng chậm, sản phẩm bị hỏng."))


In [ ]:
# Nhập văn bản tùy chỉnh để test
user_input = input("Nhập văn bản cần phân tích cảm xúc: ")
result = predict_sentiment(user_input)
print("Kết quả:", result)
